# Homework 2: BERT on AWS

In this homework, we will apply the BERT algorithm on Amazon Web Services.  

This [DataRec repository](https://github.com/sisinflab/DataRec) contains a pointer to accessing recommendation system data, installable via 

```python
pip install datarec-lib
```

**General rules of thumb for homeworks:**
- Read the homework questions carefully.
- Explain your choices.
- Present your findings concisely.
- Use tables, plots, and summary statistics to aid your presentation of findings.
- If you have an idea in mind but could not implement (in code), present the idea thoroughly and how you would have implemented the code. 

### Tasks:

For all tasks below, create one or more functions for each step such that a sequence of functions may be run for a full analysis.  Specify the sequence of functions and their brief descriptions in the README.

1. Download the MovieLens 1m dataset.  You should output a copy of the dataset on an AWS S3 bucket.  
    - Check if your S3 bucket already contains the dataset. If so, the script should not actually download the dataset.

In [63]:
#!pip install boto3 python-dotenv

In [64]:
import os
from pathlib import Path
import urllib.request
import zipfile

import boto3
from boto3.s3.transfer import S3UploadFailedError
from botocore.exceptions import ClientError
from dotenv import load_dotenv

In [65]:
# Load AWS credentials from homework_2/.env and build aws clients.
load_dotenv('.env', override=True)

session_kwargs = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
    "aws_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "aws_session_token": os.getenv("AWS_SESSION_TOKEN"),
    "region_name": os.getenv("AWS_REGION"),
}
session_kwargs = {k: v for k, v in session_kwargs.items() if v}

aws_session = boto3.Session(**session_kwargs)
bucket_name = os.getenv("AWS_S3_BUCKET_NAME", "stall-de300-winter26")
s3_client = aws_session.client("s3")
sts = aws_session.client("sts")

In [66]:
print(f"Using bucket: {bucket_name}")
print(sts.get_caller_identity())

Using bucket: stall-de300-winter26
{'UserId': 'AROAYAAO5HRMPYDMRADKI:alt5629', 'Account': '549787090008', 'Arn': 'arn:aws:sts::549787090008:assumed-role/AWSReservedSSO_mse-tl-dataeng300-EMR_da0cc2e9e5742c69/alt5629', 'ResponseMetadata': {'RequestId': '629568b2-570b-49da-b273-f08e5190aa75', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '629568b2-570b-49da-b273-f08e5190aa75', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTI6UzoxNzcxNDg1ODQ4MjQzOlI6SkxkV1g1ZHQ=', 'content-type': 'text/xml', 'content-length': '474', 'date': 'Thu, 19 Feb 2026 07:24:08 GMT'}, 'RetryAttempts': 0}}


In [67]:
MOVIELENS_1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
# Value used by many functions
MOVIELENS_DIR = Path("ml-1m")
MOVIELENS_ZIP = Path("ml-1m.zip")

In [68]:
def _get_bucket_name(arn_or_name: str) -> str:
    """Accept either a plain bucket name or an S3 bucket ARN and
    return the bucket name portion that the boto3 S3 client expects."""
    prefix = "arn:aws:s3:::"
    if arn_or_name.startswith(prefix):
        return arn_or_name[len(prefix) :]
    return arn_or_name


def download_movielens_1m() -> Path:
    """Download the MovieLens 1M dataset if it is not already present locally.

    Returns a Path to the extracted MovieLens 1M directory.
    """
    # If the directory already exists and is non-empty, assume it's usable
    if MOVIELENS_DIR.exists() and any(MOVIELENS_DIR.iterdir()):
        print(f"MovieLens 1M already present at {MOVIELENS_DIR.resolve()}")
        return MOVIELENS_DIR

    # Ensure we have the ZIP file; only download from the MovieLens URL if it is missing
    if not MOVIELENS_ZIP.exists():
        print(f"Downloading MovieLens 1M from {MOVIELENS_1M_URL} ...")
        try:
            urllib.request.urlretrieve(MOVIELENS_1M_URL, MOVIELENS_ZIP)
        except Exception as exc:
            raise RuntimeError(f"Failed to download MovieLens 1M: {exc}") from exc

    # Extract the archive (creates 'ml-1m' directory)
    print(f"Extracting {MOVIELENS_ZIP} ...")
    try:
        with zipfile.ZipFile(MOVIELENS_ZIP, "r") as zf:
            zf.extractall()
    except zipfile.BadZipFile as exc:
        raise RuntimeError(f"Corrupted MovieLens ZIP file: {exc}") from exc

    # Check that extraction actually worked
    if not MOVIELENS_DIR.exists():
        raise FileNotFoundError(
            f"Expected directory {MOVIELENS_DIR} not found after extraction."
        )

    print(f"MovieLens 1M extracted to {MOVIELENS_DIR.resolve()}")
    return MOVIELENS_DIR


def s3_bucket_contains() -> bool:
    """Check whether the configured S3 bucket already contains a copy
    of the MovieLens 1M dataset under the 'ml-1m/' prefix."""
    bucket = _get_bucket_name(bucket_name)
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix="ml-1m/")
    except ClientError as exc:
        # If we cannot list objects (e.g., permissions), treat as empty so
        # that the rest of the pipeline can attempt an upload.
        print(f"Warning: failed to list objects in bucket {bucket}: {exc}")
        return False

    has_objects = "Contents" in response and len(response["Contents"]) > 0
    if has_objects:
        print(f"S3 bucket '{bucket}' already contains MovieLens 1M data under 'ml-1m/'.")
    else:
        print(f"S3 bucket '{bucket}' does not yet contain MovieLens 1M data.")
    return has_objects


def send_to_s3_bucket():
    """Upload the local MovieLens 1M dataset to the configured S3 bucket
    under the 'ml-1m/' prefix. Assumes the dataset has already been
    downloaded locally."""
    bucket = _get_bucket_name(bucket_name)

    if not MOVIELENS_DIR.exists() or not any(MOVIELENS_DIR.iterdir()):
        raise FileNotFoundError(
            f"Local MovieLens directory '{MOVIELENS_DIR}' does not exist or is empty. "
            "Run download_movielens_1m() first."
        )

    print(f"Uploading contents of {MOVIELENS_DIR.resolve()} to s3://{bucket}/ml-1m/ ...")
    for local_path in MOVIELENS_DIR.rglob("*"):
        if not local_path.is_file():
            continue

        # Construct an S3 key that preserves the relative directory structure.
        relative_path = local_path.relative_to(MOVIELENS_DIR)
        s3_key = f"ml-1m/{relative_path.as_posix()}"

        try:
            s3_client.upload_file(
                Filename=str(local_path),
                Bucket=bucket,
                Key=s3_key,
            )
            print(f"Uploaded {local_path} -> s3://{bucket}/{s3_key}")
        except (S3UploadFailedError, ClientError) as exc:
            print(f"Failed to upload {local_path} to s3://{bucket}/{s3_key}: {exc}")

In [69]:
def orchestrate_movielens_download():
    download_movielens_1m()
    if s3_bucket_contains():
        return
    send_to_s3_bucket()

In [70]:
orchestrate_movielens_download()

Extracting ml-1m.zip ...
MovieLens 1M extracted to C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m
S3 bucket 'stall-de300-winter26' does not yet contain MovieLens 1M data.
Uploading contents of C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m to s3://stall-de300-winter26/ml-1m/ ...
Uploaded ml-1m\movies.dat -> s3://stall-de300-winter26/ml-1m/movies.dat
Uploaded ml-1m\ratings.dat -> s3://stall-de300-winter26/ml-1m/ratings.dat
Uploaded ml-1m\README -> s3://stall-de300-winter26/ml-1m/README
Uploaded ml-1m\users.dat -> s3://stall-de300-winter26/ml-1m/users.dat


2. Create embeddings for the BERT algorithm.  For the same set of items (movies) in the dataset, you should create the embeddings once and output a copy of the necessary intermediate results on the S3 bucket.
    - This is the *offline* step, where embeddings only need to be created once for the recommendation system.
    - Use a random subset (30%) of users in the available dataset.

In [71]:
#!pip install transformers

In [72]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import pandas as pd

In [73]:
def read_ml1m() -> tuple[pd.DataFrame]:
    ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", engine="python",
                      names=["user_id","movie_id","rating","timestamp"])
    movies  = pd.read_csv("ml-1m/movies.dat",  sep="::", engine="python",
                      names=["movie_id","title","genres"], encoding="latin-1")
    users   = pd.read_csv("ml-1m/users.dat", sep="::", engine="python",
                      names=["user_id", "gender", "age", "occupation", "zip"])
    return ratings,movies,users

ratings, movies, users = read_ml1m()
ratings.head(), movies.head(), users.head()

(   user_id  movie_id  rating  timestamp
 0        1      1193       5  978300760
 1        1       661       3  978302109
 2        1       914       3  978301968
 3        1      3408       4  978300275
 4        1      2355       5  978824291,
    movie_id                               title                        genres
 0         1                    Toy Story (1995)   Animation|Children's|Comedy
 1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
 2         3             Grumpier Old Men (1995)                Comedy|Romance
 3         4            Waiting to Exhale (1995)                  Comedy|Drama
 4         5  Father of the Bride Part II (1995)                        Comedy,
    user_id gender  age  occupation    zip
 0        1      F    1          10  48067
 1        2      M   56          16  70072
 2        3      M   25          15  55117
 3        4      M   45           7  02460
 4        5      M   25          20  55455)

In [74]:
def sample_users(users_df: pd.DataFrame, sample_pct: float) -> pd.DataFrame:
    """
    given a dataframe of users from the ml-1m dataset, returns a randomly sampled
    dataframe of `sample_pct`% of the users from `users_df`.
    """
    if not 0 < sample_pct <= 100:
        raise ValueError("sample_pct must a number >0 and <=100, got ", sample_pct)
    sample_frac = sample_pct / 100.0
    return users_df.sample(frac=sample_frac, random_state=42)

# I made this function so I can keep my sample consistent when generating embeddings, to
# be able to reuse my work for later tasks.
def sample_or_load_users(sample_pct: float, users_df: pd.DataFrame = users) -> pd.DataFrame:
    """
    Attempts to load a sampled users file from ml-1m/users_sample_{sample_pct}.csv.
    If it does not exist, samples users_df and saves it to that path.
    """
    sample_file = MOVIELENS_DIR / f"users_sample_{int(sample_pct)}.csv"
    if sample_file.exists():
        print(f"{sample_file} already exists, loading df from csv")
        return pd.read_csv(sample_file)

    sampled_df = sample_users(users_df, sample_pct)
    sampled_df.to_csv(sample_file, index=False)
    return sampled_df

In [75]:
users_sample: pd.DataFrame = sample_or_load_users(30, users)
print(users.count())
print(users_sample.count())


user_id       6040
gender        6040
age           6040
occupation    6040
zip           6040
dtype: int64
user_id       1812
gender        1812
age           1812
occupation    1812
zip           1812
dtype: int64


In [76]:
def get_bert_pretrained():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MODEL_NAME = "distilbert-base-uncased" # for illustrations, 66M model
    print(DEVICE, " | ", MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    return tokenizer, encoder

tokenizer, encoder = get_bert_pretrained()
encoder.eval()

cpu  |  distilbert-base-uncased


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [77]:
# ----------------------------
# 3) Offline job: movie embeddings for ml-1m
# ----------------------------

# taken and lightly modified from lab 4
@torch.no_grad
def encode_texts_in_batches(texts, batch_size=128, max_len=64):
    """Encode a list of texts into L2-normalized CLS embeddings."""
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inp = tokenizer(
            batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        ).to(DEVICE)
        out = encoder(**inp).last_hidden_state[:, 0, :]
        embs.append(out.cpu())
    embs = torch.cat(embs, dim=0)
    return F.normalize(embs, p=2, dim=1)

def create_movie_embeddings(movies: pd.DataFrame) -> F.Tensor:
    movies["text"] = movies["title"].fillna("") + " [SEP] " + movies["genres"].fillna("")
    item_emb = encode_texts_in_batches(movies["text"].tolist(), batch_size=128, max_len=64)
    print("Item embedding matrix shape:", item_emb.shape)
    return item_emb


In [78]:
# expensive step
item_emb = create_movie_embeddings(movies)

Item embedding matrix shape: torch.Size([3883, 768])


In [79]:
# Build positive interactions restricted to the sampled 30% of users like we did in lab
def build_positive_ratings(ratings, users_sample):
    ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
    user_subset = set(users_sample["user_id"].tolist())
    sample_ratings = ratings[ratings["user_id"].isin(user_subset)]
    pos = sample_ratings[sample_ratings["rating"] >= 4].copy()
    pos["value"] = 1.0
    pos = pos.sort_values(["user_id", "timestamp"])

    events = pos[["user_id", "movie_id", "timestamp"]].rename(columns={"timestamp": "ts"})
    print(
        f"Sample users: {len(user_subset)} | positive interactions: {len(pos)} | unique movies: {pos['movie_id'].nunique()}"
    )

    return events, sample_ratings


In [80]:
events, sample_ratings = build_positive_ratings(ratings, users_sample)

Sample users: 1812 | positive interactions: 170318 | unique movies: 3275


In [81]:
# Build Faiss index for ANN search using normalized embeddings
def build_faiss_index(item_emb):
    index = faiss.IndexFlatIP(item_emb.shape[1])
    index.add(item_emb.numpy().astype("float32"))
    print("Faiss index size:", index.ntotal)
    return index


In [82]:
index = build_faiss_index(item_emb)

Faiss index size: 3883


In [83]:
# Helper to upload any artifact to the configured S3 bucket
def try_upload_artifact(local_path: Path, s3_key: str, override: bool=False):
    """Upload a local file to S3, skipping if the key already exists in the bucket.
    If `override` = `true`, upload anyways."""
    bucket = _get_bucket_name(bucket_name)
    if not local_path.exists():
        raise FileNotFoundError(f"Missing artifact: {local_path}")
    
    # Check if the S3 key already exists
    try:
        s3_client.head_object(Bucket=bucket, Key=s3_key)
        if override:
            print(f"S3 key '{s3_key}' already exists in bucket '{bucket}', attempting upload anyways.")
        else:
            print(f"S3 key '{s3_key}' already exists in bucket '{bucket}', skipping upload.")
            return
    except ClientError as e:
        # If a 404 error, the object does not exist, so proceed with upload
        if e.response['Error']['Code'] == '404':
            pass
        else:
            # Some other error occurred
            raise
    
    print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
    s3_client.upload_file(str(local_path), bucket, s3_key)

# Persist offline artifacts locally and to S3
def save_embeddings_to_s3(item_emb, index, movies: pd.DataFrame, bucket_name: str):
    MOVIELENS_DIR.mkdir(exist_ok=True)
    emb_path = MOVIELENS_DIR / "item_emb_with_ids.pt"
    index_path = MOVIELENS_DIR / "item_emb.index"
    sample_users_path = MOVIELENS_DIR / "users_sample_30.csv"

    payload = {
        "item_emb": item_emb,
        "movie_id": movies["movie_id"].to_numpy(),
        "text": movies["text"].to_numpy(),
    }
    torch.save(payload, emb_path)
    faiss.write_index(index, str(index_path))

    print(f"Saved embeddings to {emb_path.resolve()}")
    print(f"Saved faiss index to {index_path.resolve()}")
    print(f"Sampled users stored at {sample_users_path.resolve()}")

    # from the local files, upload embeddings, embedding index, and sample user csv
    try_upload_artifact(emb_path, f"ml-1m/{emb_path.name}")
    try_upload_artifact(index_path, f"ml-1m/{index_path.name}")
    try_upload_artifact(sample_users_path, "ml-1m/users_sample_30.csv")



In [84]:
save_embeddings_to_s3(item_emb, index, movies, bucket_name)

Saved embeddings to C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m\item_emb_with_ids.pt


Saved faiss index to C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m\item_emb.index
Sampled users stored at C:\Users\Natha\source\repos\Classes\DE_300\homework\homework_2\ml-1m\users_sample_30.csv
Uploading ml-1m\item_emb_with_ids.pt -> s3://stall-de300-winter26/ml-1m/item_emb_with_ids.pt
Uploading ml-1m\item_emb.index -> s3://stall-de300-winter26/ml-1m/item_emb.index
Uploading ml-1m\users_sample_30.csv -> s3://stall-de300-winter26/ml-1m/users_sample_30.csv


3. Recommend five movies for each of the following users.  The recommendations should be saved in a file on the S3 bucket containing `User_Type`, `Last_Interaction_Time`, other user summaries in the dataset,and a list of recommended movies:
    - *Cold user*: a user that the system has no data on.
    - *Top user*: a random user who has frequently rated movies (number of interactions among the top 5\% of users).

In [85]:
# cold user: not in the sampled user set
def get_cold_user(users, users_sample):
    all_user_ids = set(users["user_id"])
    user_subset = set(users_sample["user_id"].tolist())
    cold_user_candidates = all_user_ids - user_subset
    cold_user = sorted(cold_user_candidates)[0]  # pick first user for consistency
    print(f"Cold user: {cold_user}")
    return cold_user

# identify a top user: in the top 5% by number of interactions
def get_top_user(sample_ratings): 
    user_interaction_counts = sample_ratings.groupby("user_id").size().reset_index(name="interaction_count")
    top_5_percent_threshold = user_interaction_counts["interaction_count"].quantile(0.95)
    top_users = user_interaction_counts[user_interaction_counts["interaction_count"] >= top_5_percent_threshold]
    top_user = top_users.sample(1, random_state=42)["user_id"].iloc[0]
    print(f"Top user: {top_user} with {user_interaction_counts[user_interaction_counts['user_id']==top_user]['interaction_count'].iloc[0]} interactions")
    return top_user

cold_user = get_cold_user(users, users_sample)
top_user = get_top_user(sample_ratings)

Cold user: 1
Top user: 1943 with 680 interactions


In [86]:
# Helpers for task 3 recommendations (sampled 30% users)

def _encode_single_text(text: str, max_len: int = 128) -> np.ndarray:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if not text:
        return None
    with torch.no_grad():
        batch = tokenizer(
            [text], padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        ).to(DEVICE)
        out = encoder(**batch).last_hidden_state[:, 0, :]
        emb = F.normalize(out, p=2, dim=1).cpu().numpy().astype("float32")
        return emb


def build_user_history(user_id: int, events_df: pd.DataFrame, movies_df: pd.DataFrame, n: int = 10):
    user_events = events_df[events_df["user_id"] == user_id].sort_values("ts")
    last_ts = user_events["ts"].max() if not user_events.empty else None
    seen = set(user_events["movie_id"].tolist())
    texts = []
    if not user_events.empty:
        recent = user_events.tail(n)["movie_id"].tolist()
        texts = movies_df.set_index("movie_id").loc[recent, "text"].fillna("").tolist()
    return texts, seen, last_ts


# taken and modified from lab 4, used by task-specific recommender functions
def recommend(
    user_id: int,
    movies_df: pd.DataFrame,
    events_df: pd.DataFrame,
    idx: faiss.IndexFlatIP,
    k: int = 5,
    history_n: int = 10, # most recent user ratings
    fallback_ratings: pd.DataFrame | None = None,
    popular_min_rating: int = 4,
):
    """Shared recommender: uses history when present, otherwise popularity fallback."""
    texts, seen, last_ts = build_user_history(user_id, events_df, movies_df, n=history_n)

    if texts:
        user_text = " ".join(texts)
        u = _encode_single_text(user_text)
        scores, idxs = idx.search(u, k + len(seen) + 20)
        recs = []
        for j in idxs[0]:
            mid = int(movies_df.iloc[j]["movie_id"])
            if mid in seen:
                continue
            recs.append(mid)
            if len(recs) == k:
                break
    else:
        recs = []
        if fallback_ratings is not None:
            top_popular = (
                fallback_ratings[fallback_ratings["rating"] >= popular_min_rating]
                .groupby("movie_id")["rating"]
                .size()
                .sort_values(ascending=False)
                .head(k)
                .index.tolist()
            )
            recs = top_popular
    return {"user_id": user_id, "last_ts": last_ts, "recs": recs, "seen": list(seen)}


def recommend_for_user(user_id: int, sample_ratings: pd.DataFrame, k: int = 5):
    # wrapper for sampled users/events/index
    return recommend(
        user_id=user_id,
        movies_df=movies,
        events_df=events,
        idx=index,
        k=k,
        history_n=10,
        fallback_ratings=sample_ratings,
    )


def summarize_user(user_id: int, user_type: str, sample_ratings: pd.DataFrame):
    rec_info = recommend_for_user(user_id, sample_ratings, k=5)
    row = users[users["user_id"] == user_id]
    summary = {
        "User_Type": user_type,
        "User_ID": user_id,
        "Last_Interaction_Time": rec_info["last_ts"],
        "Interactions": len(rec_info["seen"]),
        "Recs": rec_info["recs"],
    }
    if not row.empty:
        summary.update({
            "Gender": row.iloc[0]["gender"],
            "Age": row.iloc[0]["age"],
            "Occupation": row.iloc[0]["occupation"],
            "Zip": row.iloc[0]["zip"],
        })
    return summary


def orchestrate_task3():
    cold_user = get_cold_user(users, users_sample)
    top_user = get_top_user(sample_ratings)
    records = [summarize_user(cold_user, "cold", sample_ratings), summarize_user(top_user, "top", sample_ratings)]
    df = pd.DataFrame(records)
    out_path = MOVIELENS_DIR / "task3_recommendations.csv"
    df.to_csv(out_path, index=False)
    try_upload_artifact(out_path, "ml-1m/task3_recommendations.csv")
    return df

In [87]:
task3_recs = orchestrate_task3()
task3_recs.head()

Cold user: 1
Top user: 1943 with 680 interactions
Uploading ml-1m\task3_recommendations.csv -> s3://stall-de300-winter26/ml-1m/task3_recommendations.csv


,User_Type,User_ID,Last_Interaction_Time,Interactions,Recs,Gender,Age,Occupation,Zip
0,cold,1,NaT,0,"[2858, 260, 1196, 2028, 1198]",F,1,10,48067
1,top,1943,2002-12-14 18:38:08,424,"[2053, 2038, 3588, 2631, 2765]",M,18,4,91501


In [88]:
# Extract movie titles for the recommended movie IDs
def get_movie_titles(movie_ids, movies_df):
    """Look up movie_ids in movies_df to get movie titles"""
    movie_lookup = movies_df.set_index('movie_id')['title'].to_dict()
    return [movie_lookup.get(mid, f"Unknown ({mid})") for mid in movie_ids]

# Add a column with movie titles for the recommendations
task3_recs['Rec_Titles'] = task3_recs['Recs'].apply(lambda x: get_movie_titles(x, movies))

# Display the results
[print(recs) for recs in task3_recs['Rec_Titles']]

['American Beauty (1999)', 'Star Wars: Episode IV - A New Hope (1977)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Saving Private Ryan (1998)', 'Raiders of the Lost Ark (1981)']
['Honey, I Blew Up the Kid (1992)', 'Cat from Outer Space, The (1978)', 'King of Marvin Gardens, The (1972)', 'Frogs for Snakes (1998)', 'Acid House, The (1998)']


[None, None]

4. Repeat steps 2 and 3 but with the full set of data.  You should be able to reuse your work from earlier.

In [89]:
def build_full_embeddings(emb_path, idx_path):
    # Check if embeddings and index already exist
    if emb_path.exists() and idx_path.exists():
        payload = torch.load(emb_path)
        embs = payload["item_emb"]
        idx = faiss.read_index(str(idx_path))
        movies_full = movies.copy()
        movies_full["text"] = movies_full["title"].fillna("") + " [SEP] " + movies_full["genres"].fillna("")
        return movies_full, embs, idx
    
    # same text generation for movies df
    movies_full = movies.copy()
    movies_full["text"] = movies_full["title"].fillna("") + " [SEP] " + movies_full["genres"].fillna("")

    # same embs helper function
    embs = encode_texts_in_batches(movies_full["text"].tolist(), batch_size=128, max_len=64)

    # similar index creation
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs.numpy().astype("float32"))

    # save these full torch embeddings
    payload = {
        "item_emb": embs,
        "movie_id": movies_full["movie_id"].to_numpy(),
        "text": movies_full["text"].to_numpy(),
    }
    torch.save(payload, emb_path)
    faiss.write_index(idx, str(idx_path))
    return movies_full, embs, idx


def recommend_for_user_full(user_id: int, movies_df: pd.DataFrame, embs: torch.Tensor, idx: faiss.IndexFlatIP, k: int = 5):
    # rebuild dataframe of positive rating events
    ratings_full = ratings.copy()
    ratings_full["timestamp"] = pd.to_datetime(ratings_full["timestamp"], unit="s")
    events_full = ratings_full[ratings_full["rating"] >= 4][["user_id", "movie_id", "timestamp"]].rename(columns={"timestamp": "ts"})

    return recommend(
        user_id=user_id,
        movies_df=movies_df,
        events_df=events_full,
        idx=idx,
        k=k,
        history_n=10,
        fallback_ratings=ratings_full,
    )


def summarize_user_full(user_id: int, user_type: str, embs_full: torch.Tensor, idx):
    """Get recommendations for a user and produce a summary of the user's traits
    and recommended movies. Similar to `summarize_user()`."""
    rec_info = recommend_for_user_full(user_id, movies, embs_full, idx, k=5)
    row = users[users["user_id"] == user_id]
    summary = {
        "User_Type": user_type,
        "User_ID": user_id,
        "Last_Interaction_Time": rec_info["last_ts"],
        "Interactions": len(rec_info["seen"]),
        "Recs": rec_info["recs"],
    }
    if not row.empty:
        summary.update({
            "Gender": row.iloc[0]["gender"],
            "Age": row.iloc[0]["age"],
            "Occupation": row.iloc[0]["occupation"],
            "Zip": row.iloc[0]["zip"],
        })
    return summary


def orchestrate_task4_full():
    FULL_EMB_PATH = MOVIELENS_DIR / "item_emb_full_with_ids.pt"
    FULL_INDEX_PATH = MOVIELENS_DIR / "item_emb_full.index"
    FULL_REC_PATH = MOVIELENS_DIR / "task4_recommendations.csv"

    movies_full, embs_full, idx_full = build_full_embeddings(FULL_EMB_PATH, FULL_INDEX_PATH)
    # reuse the top user definition on full data
    ratings_full = ratings.copy()
    user_counts = ratings_full.groupby("user_id").size().reset_index(name="interaction_count")
    thresh = user_counts["interaction_count"].quantile(0.95)
    top_users_full = user_counts[user_counts["interaction_count"] >= thresh]
    top_user_full = top_users_full.sample(1, random_state=42)["user_id"].iloc[0]

    # the 1M dataset says every user has at least 20 interactions?
    # so if there are no cold users, just going to pick a random one basically
    cold_candidates = set(ratings_full["user_id"].unique()) - set(users_sample["user_id"].unique())
    cold_user_full = sorted(cold_candidates)[0] if cold_candidates else int(user_counts.sample(1, random_state=99)["user_id"].iloc[0])

    # recommend cold and top users movies with the full dataset's worth of embeddings, save to csv
    recs = [
        summarize_user_full(cold_user_full, "cold", embs_full, idx_full),
        summarize_user_full(top_user_full, "top", embs_full, idx_full)
    ]
    df = pd.DataFrame(recs)
    df.to_csv(FULL_REC_PATH, index=False)

    # try to upload full rec results to s3 if they do not already exist
    try_upload_artifact(FULL_REC_PATH, "ml-1m/task4_recommendations.csv", override=True)
    try_upload_artifact(FULL_EMB_PATH, f"ml-1m/{FULL_EMB_PATH.name}")
    try_upload_artifact(FULL_INDEX_PATH, f"ml-1m/{FULL_INDEX_PATH.name}")
    print(df)

In [90]:
orchestrate_task4_full()

Uploading ml-1m\task4_recommendations.csv -> s3://stall-de300-winter26/ml-1m/task4_recommendations.csv
Uploading ml-1m\item_emb_full_with_ids.pt -> s3://stall-de300-winter26/ml-1m/item_emb_full_with_ids.pt
Uploading ml-1m\item_emb_full.index -> s3://stall-de300-winter26/ml-1m/item_emb_full.index
  User_Type  User_ID Last_Interaction_Time  Interactions  \
0      cold        1   2001-01-06 23:39:11            45   
1       top     3519   2003-02-15 22:38:13           199   

                            Recs Gender  Age  Occupation    Zip  
0    [2354, 1064, 239, 364, 661]      F    1          10  48067  
1  [1841, 1146, 847, 2417, 3174]      F   25          14  11215  


5. Choose and rate 10 movies and create a "user profile" for yourself.  Save your user profile on the S3 bucket.  Recommend 5 movies for yourself and save the results on the S3 bucket.

In [91]:
# create a personal profile, recommend 5 movies, and upload to s3


def create_self_profile():
    SELF_USER_ID = 999999 # my fake user id
    SELF_PROFILE_PATH = MOVIELENS_DIR / "task5_self_profile.csv"

    my_movie_ratings = [
        # (id, rating)
        (1, 5),    # Toy Story (1995)
        (2571, 5), # Matrix
        (318, 5),  # Shawshank Redemption
        (44, 4),   # Mortal Kombat
        (104, 4),  # Happy Gilmore
        (123, 5),  # ChungKing Express
        (165, 5),  # Die Hard: With a Vengeance
        (2153, 5), # Avengers
        (2214, 2), # Number Seventeen
        (1704, 5), # Good Will Hunting
        (1717, 1), # Scream 2
    ]

    df = pd.DataFrame(my_movie_ratings, columns=["movie_id", "rating"])
    df["user_id"] = SELF_USER_ID
    df["timestamp"] = pd.Timestamp.utcnow()
    df.to_csv(SELF_PROFILE_PATH, index=False)
    try_upload_artifact(SELF_PROFILE_PATH, "ml-1m/task5_self_profile.csv", override=True)
    return df


def recommend_for_self(k: int = 5):
    SELF_USER_ID = 999999 # my fake user id
    SELF_RECS_PATH = MOVIELENS_DIR / "task5_self_recommendations.csv"
    
    df = create_self_profile()
    liked = df[df["rating"] >= 4]
    # Build minimal events_df from the liked movies to reuse generic recommender
    events_self = liked[["user_id", "movie_id", "timestamp"]].rename(columns={"timestamp": "ts"})

    rec_info = recommend(
        user_id=SELF_USER_ID,
        movies_df=movies,
        events_df=events_self,
        idx=index,
        k=k,
        history_n=10,
        fallback_ratings=ratings,  # fallback to popular-from-full if somehow no likes
    )

    rec_df = pd.DataFrame({"user_id": [SELF_USER_ID]*len(rec_info["recs"]), "recommended_movie_id": rec_info["recs"]})
    rec_df.to_csv(SELF_RECS_PATH, index=False)
    print(rec_df)
    return rec_df


def orchestrate_task5():
    """construct a user profile, recommend 5 new movies, save results to csv and S3."""
    self_recs = recommend_for_self(k=5) # also saves to csv
    SELF_RECS_PATH = MOVIELENS_DIR / "task5_self_recommendations.csv"
    try_upload_artifact(SELF_RECS_PATH, "ml-1m/task5_self_recommendations.csv", override=True)
    return self_recs

In [92]:
self_recs = orchestrate_task5()

Uploading ml-1m\task5_self_profile.csv -> s3://stall-de300-winter26/ml-1m/task5_self_profile.csv
   user_id  recommended_movie_id
0   999999                    10
1   999999                  2722
2   999999                   172
3   999999                  1744
4   999999                  1573
Uploading ml-1m\task5_self_recommendations.csv -> s3://stall-de300-winter26/ml-1m/task5_self_recommendations.csv


# Submission guidelines
Your submission should be contained in a `homework_2` folder of your Github repository, and it should include 
- a `readme.md` file including how to run the code and what your expected outputs are (if the code is run), 
- your source code, and/or
- a `.pdf` or `.html` file containing any necessary observations and details.
    - If you find your source code self-explanatory, you may opt to skip the `.pdf` or `.html` file in this homework.


# Generative AI disclosure

*Syllabus* policy: 

Required disclosure: each submission must include an AI Usage note stating: (1) tool(s) used, (2) the key prompt(s), and (3) what you changed and how you verified the results. If none, write: “AI Usage: None.”

## Tools
`ChatGPT Codex 5.2`
`Copilot in VS Code`

## Key Prompts
**Task 1**
```
I have started three functions for task 1 in this file. First, I need to download the MovieLens 1M dataset from "https://files.grouplens.org/datasets/movielens/ml-1m.zip". If I already downloaded it, skip this step. This functionality is in download_movielens_1m(). 

Next, I need to check if my s3 bucket contains a copy of this dataset already. This functionality should be in s3_bucket_contains(). If it does not exist, please upload it to s3 in the send_to_s3_bucket() function. All 3 of these functions will be wrapped in orchestrate_movielens_download().

Use dotenv to load credentials and boto3 for the aws clients.
```

(3) I fixed up the code to make sure that the bucket names and paths were correct, and that the logic for skipping downloads or uploads worked properly if the files already existed. I then verified that the code worked by putting in the appropriate environment variables and trying to download/upload multiple times, making sure that missing files were correctly retrieved, and nothing was re-uploaded when it already existed.

**Task 2**

```
Please complete sample_or_load_users(), which attempts to load a sample of users from ml-1m/users_sample_{sample_pct} file, and if that file does not exist, will instead call the sample_users( ) function I made to create a new sampled dataframe and save it back to that file location.
```

```
Modify upload_artifact() to only upload if the s3 key does not already exist.
```

(3) They were very simple utility functions that I just needed to knock out in 10 seconds. Once I ran them it was very clear that they worked. 

**Task 3**

**Task 4**
```
Please modify build_full_embeddings() to try to load the embeddings from the specified path if they already exist instead of creating them.
```

```
Please adapt the recommend function from 04c-bert-example.py to a basic recommend function I can use here. The function needs to use the embedding functions I defined previously.
```

(3) As I was developing flows for each of the tasks, I had small inconsistencies in the data I was storing/passing to generate recommendations for each case. I quickly had Codex refactor this to use a standard recommend function that contained most of the core logic from the lab I was already using in a few different places.

**Task 5**
```
Go through my {homework_2} file and compile a list of the main functions I call in order. I need to list the crucial components of the pipeline.
```

(3) this was just a time saver. I could have gone through the file and listed the remaining functions myself, but there was no reason not to generate the summary with AI. I verified that functions were all listed in the right order and that the descriptions made sense, but it was faster than writing it all myself. 